# 01 — Run Experiments (pi0.5, LIBERO / LIBERO-PRO)

**This notebook IS the experiment.** The `pnp` package provides primitives
(`run_episode`, `iter_task_envs`, `store`); the loop + the `METHODS` dict below are the
visible spec of what runs. Edit the flags, re-run. Results go to Supabase.

## 1. Secrets + install (clone private repo, editable)

In [ ]:
import os
from google.colab import userdata
for k in ('SUPABASE_URL', 'SUPABASE_SERVICE_KEY', 'HF_TOKEN'):
    os.environ[k] = userdata.get(k)
GH_PAT = userdata.get('GH_PAT')
REPO_DIR = '/content/cs159-sp26'
GIT_REF = 'main'  # Change to 'main' after the refactor is merged.
![ -d "$REPO_DIR/.git" ] || git clone -q --branch "$GIT_REF" https://$GH_PAT@github.com/ArjunS07/cs159-sp26.git "$REPO_DIR"
!git -C "$REPO_DIR" fetch -q origin "$GIT_REF"
!git -C "$REPO_DIR" checkout -q "$GIT_REF"
!git -C "$REPO_DIR" pull -q --ff-only origin "$GIT_REF"
!pip install -q -e "$REPO_DIR/pnp-vla[sim]"

## 2. Environment + model + store

In [ ]:
from pnp.env_setup import setup_environment
setup_environment()

In [ ]:
from pnp import libero_env, models, RolloutConfig, Method
from pnp.store import SupabaseStore
from pnp.rollout import run_episode, iter_task_envs

libero_env.init_libero_benchmark()
policy, preprocess, postprocess = models.load_pi05()
device = models.default_device()
store = SupabaseStore()
episodes = libero_env.build_final_episodes()   # 8 stock tasks x 10 episodes

## 3. The experiment: controlled 80-episode slice

`METHODS` is the whole spec. `pnp_uncertainty_only` is the RNG-isolated no-op baseline;
`extra_steps` is the matched-compute baseline; refinement is the intervention.

In [ ]:
EXPERIMENT = 'slice-v1'
S, K = (2, 3), 3

# METHODS is a LIST of (method, cfg) — a list (not a dict) so the two refinement variants can
# share method=Method.REFINEMENT; they differ only by the refine_average column (and so get
# distinct rollout_ids). Method strings come from pnp.config.Method — the same source analysis/
# filters on, so labels never drift.
#   vanilla     : no probe, base sampler
#   extra_steps : matched-compute baseline (more Euler steps)
#   uncertainty : probe set, no action -> RNG-isolated no-op that MEASURES uncertainty
#   refinement  : re-noise from the probe's clean estimate (last, then averaged variant)
METHODS = [
    (Method.VANILLA,     RolloutConfig()),
    (Method.EXTRA_STEPS, RolloutConfig(num_inference_steps=16)),
    (Method.UNCERTAINTY, RolloutConfig(pnp_steps=S, pnp_k=K)),
    (Method.REFINEMENT,  RolloutConfig(pnp_steps=S, pnp_k=K, refine=True)),
    (Method.REFINEMENT,  RolloutConfig(pnp_steps=S, pnp_k=K, refine=True, refine_average=True)),
]

store.start_run(driver='slice', benchmark='libero', experiment=EXPERIMENT)
done = store.existing_keys(EXPERIMENT)
n = 0
for env, task_eps in iter_task_envs(episodes):                 # repo: env lifecycle
    for ep, name, cfg, rid in store.iter_todo(EXPERIMENT, task_eps, METHODS, done=done):
        res = run_episode(env, ep, policy, preprocess, device, cfg)
        store.log_result(rid, ep, name, cfg, res)
        n += 1
store.finish_run(n_rollouts=n)
print(f'logged {n} rollouts to experiment={EXPERIMENT}  (~{store.bytes_written/1e6:.1f} MB blobs)')

## 4. LIBERO-PRO 600-episode stretch

Same loop shape, PRO episodes + the refinement-variant methods. Requires the LIBERO-PRO
assets + `pnp.libero_pro` setup (run the LIBERO-PRO setup notebook first). Uncomment to run.

In [ ]:
# from pnp import libero_pro
# libero_pro.apply_env_patches(); libero_pro.patch_torch_load()
# bd = libero_pro.reload_benchmark()
# pro_eps = libero_pro.build_libero_pro_episodes(bd)
#
# PRO_EXPERIMENT = 'pro-v1'
# PRO_S, PRO_K = (3, 4), 10
# probe = dict(pnp_steps=PRO_S, pnp_k=PRO_K, compute_multimodal=True)   # geometry needs k>=4
# PRO_METHODS = [
#     (Method.UNCERTAINTY, RolloutConfig(**probe)),
#     (Method.REFINEMENT,  RolloutConfig(**probe, refine=True)),
#     (Method.REFINEMENT,  RolloutConfig(**probe, refine=True, refine_average=True)),
# ]
# store.start_run(driver='run_pro', benchmark='libero_pro', experiment=PRO_EXPERIMENT)
# done = store.existing_keys(PRO_EXPERIMENT)
# for env, task_eps in iter_task_envs(pro_eps):
#     for ep, name, cfg, rid in store.iter_todo(PRO_EXPERIMENT, task_eps, PRO_METHODS, done=done):
#         res = run_episode(env, ep, policy, preprocess, device, cfg)
#         store.log_result(rid, ep, name, cfg, res)
# store.finish_run()